In [ ]:
%pip install langchain_openai

In [ ]:
%pip install duckduckgo-search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 43.8 MB/s eta 0:00:00


In [ ]:
from duckduckgo_search import DDGS

# Переопределяем функцию web_search для использования DuckDuckGo
# Инструмент для поиска в интернете, использующий DuckDuckGo
def web_search(query: str) -> str:
    """Ищет информацию в интернете по заданному запросу, используя DuckDuckGo."""
    print(f"Performing web search for: {query} using DuckDuckGo")
    try:
        with DDGS() as ddgs:
            results = [r for r in ddgs.text(query, max_results=5)] # Получаем до 5 результатов
        if results:
            formatted_results = "\n".join([f"Заголовок: {r['title']}\nURL: {r['href']}\nОписание: {r['body']}\n" for r in results])
            return formatted_results
        else:
            return f"По запросу '{query}' ничего не найдено."
    except Exception as e:
        return f"Ошибка при поиске в интернете через DuckDuckGo: {e}"

print("Функция web_search обновлена для использования DuckDuckGo.")


Функция web_search обновлена для использования DuckDuckGo.


In [ ]:
SYSTEM_PROMPT = """
Ты автономный агент.

Ты работаешь строго в JSON формате:

{
 "thought": "...",
 "action": "tool_name OR finish",
 "action_input": "...", // Ожидаемый ввод для инструмента, или null/пустая строка, если action = finish
 "answer": "..."   // только если action = finish
}

Инструменты:
- calculator(expression: str): Вычисляет математическое выражение.
- web_search(query: str): Ищет информацию в интернете по заданному запросу, используя DuckDuckGo.
- write_file(file_path: str, content: str): Записывает содержимое в текстовый файл.
"""


In [ ]:
%pip install langchain-groq

In [ ]:
%pip install DDGS

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:
import json

In [ ]:
from langchain_groq import ChatGroq

# Передаем ключ напрямую в аргумент api_key
model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key="key"
)

In [ ]:
# STATE

In [ ]:
class AgentState:
  def __init__(self) -> None:
    self.messages=[]
    self.steps=[]
    self.step_count=0
    self.max_steps=8
    self.finished=False
    self.final_answer=None




In [ ]:
# Tools

In [ ]:
import datetime
import subprocess
import os
from duckduckgo_search import DDGS

def calculator(expression: str) -> str:
    """Вычисляет математическое выражение."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Ошибка: {e}"

# Инструмент для поиска в интернете, использующий DuckDuckGo
def web_search(query: str) -> str:
    """Ищет информацию в интернете по заданному запросу, используя DuckDuckGo."""
    print(f"Performing web search for: {query} using DuckDuckGo")
    try:
        with DDGS() as ddgs:
            results = [r for r in ddgs.text(query, max_results=5)] # Получаем до 5 результатов
        if results:
            formatted_results = "\n".join([f"Заголовок: {r['title']}\nURL: {r['href']}\nОписание: {r['body']}\n" for r in results])
            return formatted_results
        else:
            return f"По запросу '{query}' ничего не найдено."
    except Exception as e:
        return f"Ошибка при поиске в интернете через DuckDuckGo: {e}"

# Инструмент для записи файла
def write_file(file_path: str, content: str) -> str:
    """Записывает содержимое в текстовый файл."""
    try:
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(content)
        return f"Файл '{file_path}' успешно записан."
    except Exception as e:
        return f"Ошибка записи в файл '{file_path}': {e}"

print("Инструменты для агента загружены.")


Инструменты для агента загружены.


In [ ]:
TOOLS = {
    "calculator": calculator,
    "web_search": web_search,
    "write_file": write_file,
}


In [ ]:
def run_tool(name, arg):
    if name not in TOOLS:
        return f"Unknown tool: {name}"
    return TOOLS[name](arg)

In [ ]:
# Promt

In [ ]:
SYSTEM_PROMPT = """
Ты автономный агент.

Ты работаешь строго в JSON формате:

{
 "thought": "...",
 "action": "tool_name OR finish",
 "action_input": "...", // Ожидаемый ввод для инструмента, или null/пустая строка, если action = finish
 "answer": "..."   // только если action = finish
}

Инструменты:
- calculator(expression: str): Вычисляет математическое выражение.
- web_search(query: str): Ищет информацию в интернете по заданному запросу, используя LLM для генерации ответа.
- write_file(file_path: str, content: str): Записывает содержимое в текстовый файл.
"""


In [ ]:
# LLM Call

In [ ]:
def call_llm(messages):
    response = model.invoke(messages)
    return response.content

In [ ]:
# SAFE PARSER

In [ ]:
def parse_json(text):
    try:
        return json.loads(text)
    except:
        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end != -1:
            return json.loads(text[start:end+1])
        raise ValueError("Bad JSON from LLM")